# DDPM: 離散時刻のノイズ予測モデル

DDPMは、時刻ごとに少しずつガウスノイズを加える離散過程を定義し、ニューラルネットで混ぜたノイズを予測する。


## このノートの読み方

想定読者: Diffusionのforward noisingとMSE回帰を理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

Diffusionの直感を、`beta_t`、`alpha_t`、`bar_alpha_t`で計算できる形へ落とし込む。学習は`epsilon`予測、生成は逆向きsamplingである。


## 到達目標

- `alpha_t`と`bar_alpha_t`を区別できる
- `q(x_t|x_0)`の閉形式をコードにできる
- `Trainer`でノイズ予測を学習できる


## 重要語句

- `beta schedule`: 各時刻で加えるノイズ量
- `epsilon_theta`: モデルが予測するノイズ
- `reverse process`: x_Tからx_0へ戻す生成手順


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| t | (B,) | 時刻 |
| xt | (B, dim) | 時刻tのノイズ付きデータ |
| eps | (B, dim) | 正解ノイズ |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| alphaとalpha_bar | `alpha_t=1-beta_t`は1ステップの保持率、`bar_alpha_t`は時刻0からtまでの累積保持率である。 |
| 動的サンプリング | 学習では各batchごとに`t`と`epsilon`を選び直す。固定したノイズだけを覚えないようにするためである。 |
| posterior mean | 生成時は`epsilon_theta(x_t,t)`から`x_0`方向を推定し、`mu_theta`を作って`x_{t-1}`へ進む。 |
| U-Netへの導線 | ここではMLPでshapeを追う。画像では同じノイズ予測をU-Netに置き換え、時刻埋め込みを各層へ入れる。 |
| バイオ用途 | 顕微鏡画像の復元や分子構造候補生成では、ノイズからデータらしい状態へ戻す手順として読める。 |


## Forward process

各ステップで少しだけ構造を壊す。

$$
q(x_t|x_{t-1})=\mathcal{N}(\sqrt{1-\beta_t}x_{t-1},\beta_t I)
$$


## Closed form

`x_0`から任意の`x_t`を直接作れる。

$$
x_t=\sqrt{\bar{\alpha}_t}x_0+\sqrt{1-\bar{\alpha}_t}\epsilon
$$


## Simple loss

モデルは画像ではなく、混ぜたノイズを予測する。

$$
L_{\mathrm{simple}}=\mathbb{E}\|\epsilon-\epsilon_\theta(x_t,t)\|^2
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
T = 8
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
x0 = torch.ones(2, 3)
t = torch.tensor([2, 6])
eps = torch.randn_like(x0)
ab = alpha_bars[t].view(-1, 1)
xt = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps
print("alpha_bars:", alpha_bars.round(decimals=4))
print("xt:", xt.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/ddpm_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/ddpm_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/ddpm_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="DDPM: 離散時刻のノイズ予測モデル difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### beta schedule animation

- 学習目標: beta, alpha, bar_alphaの違いを見せる
- 誤解の防止: alphaとbar_alphaを混同する

対応する式:

$$
\alpha_t=1-\beta_t,\ \bar{\alpha}_t=\prod_s\alpha_s
$$


<p><a href="../demos/ddpm_beta_schedule.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/ddpm_beta_schedule.html</code>）</p>
<iframe
  src="../demos/ddpm_beta_schedule.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="beta schedule animation"
></iframe>


### x0 and epsilon to xt animation

- 学習目標: x0とepsilonの線形結合を見せる
- 誤解の防止: xtが時系列データだと思う

対応する式:

$$
x_t=\sqrt{\bar{\alpha}_t}x_0+\sqrt{1-\bar{\alpha}_t}\epsilon
$$


<p><a href="../demos/ddpm_xt_mix.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/ddpm_xt_mix.html</code>）</p>
<iframe
  src="../demos/ddpm_xt_mix.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="x0 and epsilon to xt animation"
></iframe>


### epsilon prediction animation

- 学習目標: 予測対象がnoiseであることを示す
- 誤解の防止: x0を直接予測すると思う

対応する式:

$$
\epsilon_\theta(x_t,t)
$$


<p><a href="../demos/ddpm_epsilon_prediction.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/ddpm_epsilon_prediction.html</code>）</p>
<iframe
  src="../demos/ddpm_epsilon_prediction.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="epsilon prediction animation"
></iframe>


### reverse sampling loop animation

- 学習目標: Tから0へ戻るsamplingを見せる
- 誤解の防止: 学習手順と生成手順を混同する

対応する式:

$$
x_{t-1}=\mu_\theta(x_t,t)+\sigma_t z
$$


<p><a href="../demos/ddpm_reverse_loop.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/ddpm_reverse_loop.html</code>）</p>
<iframe
  src="../demos/ddpm_reverse_loop.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="reverse sampling loop animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`DDPM: 離散時刻のノイズ予測モデル`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyDDPMDataset(Dataset):
    def __init__(self, n_samples: int = 40, dim: int = 4, steps: int = 10) -> None:
        self.x0 = torch.randn(n_samples, dim)
        self.steps = steps
        betas = torch.linspace(1e-4, 0.02, steps)
        self.alpha_bars = torch.cumprod(1 - betas, dim=0)

    def __len__(self) -> int:
        return len(self.x0)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        x0 = self.x0[index]
        t = torch.randint(0, self.steps, ())
        alpha_bar = self.alpha_bars[t]
        eps = torch.randn_like(x0)
        xt = torch.sqrt(alpha_bar) * x0 + torch.sqrt(1 - alpha_bar) * eps
        return {"xt": xt, "t": t, "eps": eps}


class TinyDDPMNoisePredictor(nn.Module):
    def __init__(self, dim: int = 4, num_steps: int = 10) -> None:
        super().__init__()
        self.num_steps = num_steps
        self.net = nn.Sequential(nn.Linear(dim + 1, 32), nn.SiLU(), nn.Linear(32, dim))

    def forward(self, xt: torch.Tensor, t: torch.Tensor, eps: torch.Tensor | None = None) -> dict[str, torch.Tensor]:
        t_embed = t.float().view(-1, 1) / max(self.num_steps - 1, 1)
        pred_eps = self.net(torch.cat([xt, t_embed], dim=-1))
        loss = torch.mean((pred_eps - eps) ** 2) if eps is not None else None
        return {"loss": loss, "logits": pred_eps}


training_args = TrainingArguments(
    output_dir="./results/ddpm_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

dataset = TinyDDPMDataset()
model = TinyDDPMNoisePredictor(num_steps=dataset.steps)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
train_output = trainer.train()
betas = torch.linspace(1e-4, 0.02, dataset.steps)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
x = torch.randn(1, 4)
with torch.no_grad():
    for t_int in reversed(range(dataset.steps)):
        t = torch.tensor([t_int])
        eps_hat = trainer.model(x, t)["logits"]
        alpha_t = alphas[t_int]
        alpha_bar_t = alpha_bars[t_int]
        beta_t = betas[t_int]
        mean = (x - beta_t / torch.sqrt(1 - alpha_bar_t) * eps_hat) / torch.sqrt(alpha_t)
        x = mean if t_int == 0 else mean + torch.sqrt(beta_t) * torch.randn_like(x)
print("DDPM Trainer loss:", train_output.training_loss)
print("reverse sample shape:", x.shape)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- reverse loopを実装する
- beta scheduleを変える
- MLPを小さなU-Netへ置き換える


## 確認問題

- `alpha_t`と`bar_alpha_t`の違いを書く。
- DDPMでモデルが予測する対象は何か。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
